# 추천 에이전트 (1) — 사용자 메모리 기반

추천의 핵심은 **사용자를 아는 것**이다. LangGraph 에이전트가 사용자 정보를 기억하고 활용하는 두 가지 방식:
1. **Short memory (State)** — 이번 대화의 State 에 담긴 사용자 정보 (이름/성별/나이 등)
2. **Long memory (Store)** — 대화를 넘어 지속되는 저장소에 쌓인 취향 (좋아함/싫어함/시청이력)

이 노트북에서 다루는 것:
- **`InjectedState`**: 도구가 LLM 이 만든 인자 대신 **State 값을 직접 주입**받는 법
- **`AgentState` 확장**: 에이전트 State 에 사용자 필드 추가
- **`InMemoryStore`**: user_id 별로 취향을 저장/조회하는 장기 메모리

> `OPENAI_API_KEY` 필요.

## 환경 변수 준비

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 가 .env 에 없습니다"

from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-4o")

## 1. `InjectedState` — 도구에 State 를 주입

보통 도구 인자는 LLM 이 생성한다. 하지만 user_id 같은 값은 LLM 이 지어내면 안 되고 **State 에서 그대로** 가져와야 한다. [basics 복습] `Annotated[State, InjectedState]` 로 선언하면 그 인자는 LLM 이 아니라 런타임 State 로 채워진다.

`AgentState`(create_react_agent 의 기본 State)를 상속해 `user_id` 필드를 추가한다.

In [ ]:
from typing import Annotated
from langgraph.prebuilt import InjectedState, create_react_agent
from langgraph.prebuilt.chat_agent_executor import AgentState

class CustomState(AgentState):
    user_id: str

def get_user_info(state: Annotated[CustomState, InjectedState]) -> str:
    """Look up user info."""
    # state 는 LLM 이 아니라 런타임에 주입됨
    user_id = state["user_id"]
    return "User is John Smith" if user_id == "user_123" else "Unknown user"

agent = create_react_agent(model=model, tools=[get_user_info], state_schema=CustomState)

In [ ]:
response = agent.invoke({"messages": "look up user information", "user_id": "user_123"})
for msg in response["messages"]:
    msg.pretty_print()

## 2. 기본 정보 기반 추천 (Short Memory)

State 에 담긴 사용자 기본 정보(이름/성별/나이)를 도구로 노출해, 이를 근거로 추천하게 한다.

In [ ]:
class InfoState(AgentState):
    user_id: str
    user_sex: str
    user_age: int

def get_user_info(state: Annotated[InfoState, InjectedState]) -> str:
    """Look up user info."""
    return f"User Info: {state['user_id']}, {state['user_sex']}, {state['user_age']}"

agent = create_react_agent(model=model, tools=[get_user_info], state_schema=InfoState)

response = agent.invoke({
    "messages": "사용자 기본 정보를 기반으로 영화를 추천해주세요.",
    "user_id": "user_123", "user_sex": "female", "user_age": 25,
})
for msg in response["messages"]:
    msg.pretty_print()

## 3. 축적된 취향 기반 추천 (Long Memory)

**`InMemoryStore`** 는 대화를 넘어 지속되는 저장소다. user_id 를 키로 취향을 넣고 뺀다. State(이번 대화)와 달리 **세션이 바뀌어도 유지**된다.

`store.put(namespace, key, value)` / `store.get(namespace, key)` 로 다룬다.

In [ ]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()
store.put(
    ("users",), "user_123",
    {"user_name": "mina", "user_sex": "female", "user_age": 25},
)
print(store.get(("users",), "user_123").value)

### 취향 조회/저장 도구

[basics 복습] 도구 안에서 `get_store()` 로 저장소에 접근하고, `config` 로 user_id 를 받는다. `save_user_info` 는 사용자가 명시적으로 취향을 말할 때만 저장하도록 설계한다.

In [ ]:
from langchain_core.runnables import RunnableConfig
from langgraph.config import get_store

def get_user_info(config: RunnableConfig) -> str:
    """Look up user info from the store."""
    store = get_store()
    user_id = config["configurable"].get("user_id")
    user_info = store.get(("users",), user_id)
    return str(user_info.value) if user_info else "Unknown user"

def save_user_info(
    likes: list[str] = None,
    dislikes: list[str] = None,
    current_watch: list[str] = None,
    config: RunnableConfig = None,
) -> str:
    """Save user preference information to the store.

    Args:
        likes: genres/movies the user likes
        dislikes: genres/movies the user dislikes
        current_watch: movies the user recently watched
    """
    store = get_store()
    user_id = config["configurable"].get("user_id")
    user_info = store.get(("users",), user_id)
    user_data = user_info.value if user_info else {}

    for field, values in [("likes", likes), ("dislikes", dislikes), ("current_watch", current_watch)]:
        if values:
            user_data.setdefault(field, []).extend(values)

    store.put(("users",), user_id, user_data)
    return (f"Updated: +{len(likes or [])} likes, +{len(dislikes or [])} dislikes, "
            f"+{len(current_watch or [])} watched.")

### 추천 에이전트 조립

[basics 복습] `create_react_agent` 에 `store` 와 `checkpointer` 를 함께 붙인다. 프롬프트로 "명시적 취향 발화일 때만 저장" 규칙을 준다 (단순 질문엔 저장 안 함).

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

RECOMMEND_PROMPT = """You are a movie recommendation assistant.
Use the user's stored preferences (liked/disliked genres, watch history) to recommend movies.
Do not mention the user's personal info (age, sex, id) directly.
If preferences are missing, use basic info (age, sex).

ONLY call save_user_info when users EXPLICITLY state preferences or watch history:
- \"I watched La La Land recently\" -> current_watch=[\"La La Land\"]
- \"I love action movies\" -> likes=[\"action\"]
- \"I didn't like The Godfather\" -> dislikes=[\"The Godfather\"]
Do NOT save when users merely ask about or discuss a movie.
Keep responses concise, warm, and in Korean.
"""

agent = create_react_agent(
    model=model,
    tools=[get_user_info, save_user_info],
    store=store,
    prompt=RECOMMEND_PROMPT,
    checkpointer=InMemorySaver(),
)

### 대화 테스트
취향을 말하면 저장되고, 이후 추천에 반영된다. (단발 호출 예시 — 실제로는 여러 턴)

In [ ]:
config = {"configurable": {"user_id": "user_123", "thread_id": "1"}}

# 1) 취향 표현 → 저장됨
r1 = agent.invoke({"messages": [{"role": "user", "content": "나 액션 영화 진짜 좋아해!"}]}, config)
r1["messages"][-1].pretty_print()

# 2) 추천 요청 → 저장된 취향 반영
r2 = agent.invoke({"messages": [{"role": "user", "content": "영화 하나 추천해줘"}]}, config)
r2["messages"][-1].pretty_print()

In [ ]:
# 저장소에 취향이 쌓였는지 확인
print(store.get(("users",), "user_123").value)

## 정리

- **Short memory (State)**: 이번 대화의 사용자 정보 → `InjectedState` 로 도구에 주입
- **Long memory (Store)**: 대화를 넘는 취향 저장소 → `InMemoryStore` + `get_store()`
- `create_react_agent(..., store=, checkpointer=)` 로 둘을 함께 활용
- 프롬프트로 "언제 저장할지" 규칙을 명시해 불필요한 저장을 막음

다음: 실제 아이템 DB + 유사도/SQL/랭킹 도구를 결합한 본격 추천 에이전트.